# DuckSeg: local analysis of one folder of recordings

Runs the full [DuckSeg](https://github.com/ronalabrcns/DuckSeg) pipeline on **your own computer**: point `DATA_DIR` to a folder with your videos and their laser-stripe masks, then run the cells from top to bottom.

**Before you start** (once per computer, see the [installation guide](https://duckseg.readthedocs.io/en/latest/installation.html)):

* Install DuckSeg and CellSAM (`pip install -e .` in the repository, then the pinned CellSAM commit).
* Accept the [CellSAM model license](https://huggingface.co/vanvalenlab/cellsam) on Hugging Face (non-commercial academic use only) and run `hf auth login`, or set `HF_TOKEN`. After the first download no token is needed.
* A CUDA GPU is strongly recommended.

The folder should contain one video and one ROI mask per experiment; all of them are analysed as one condition:

```
my_condition/
    MSH2_001.nd2        # video (.nd2, .tif or .tiff)
    MSH2_001_ROI.tif    # laser stripe mask, same size as the video
    MSH2_002.nd2
    MSH2_002_ROI.tif
    ...
```

See [Prepare your data](https://duckseg.readthedocs.io/en/latest/guides/data_preparation.html).

## 1. Your data

In [ ]:
import os

DATA_DIR = r"C:\path\to\my_condition"      # folder with the videos and *_ROI.tif masks
CONDITION_NAME = os.path.basename(os.path.normpath(DATA_DIR))   # plot label and CSV name; or e.g. "MSH2"

OUTPUT_DIR = os.path.join(DATA_DIR, "duckseg_output")   # plots and CSV are saved here

## 2. Setup

In [ ]:
import sys
sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))   # use this repository's DuckSeg if it is not installed

import torch
from DuckSeg.model_loader import load_cellsam_model
import DuckSeg.experiment_evaluator as evaluator

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (this will be slow!)")
if evaluator.cellsam_model is None:
    evaluator.cellsam_model = load_cellsam_model()   # CellSAM v1.2, downloaded on first use
print("CellSAM model loaded")

## 3. Load the recordings

All videos are cut to the shortest one, so that every cell shares the same time axis.

In [ ]:
from DuckSeg.experiment_loader import ExperimentBatch

if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError(f"Folder not found: {DATA_DIR}  (set DATA_DIR in section 1)")
batch = ExperimentBatch(CONDITION_NAME, DATA_DIR)
experiments = batch.experiments()
if not experiments:
    raise FileNotFoundError(f"No videos with a matching *_ROI.tif mask found in {DATA_DIR}")
print(f"{len(experiments)} experiments:", [e.name for e in experiments])

batch.max_len = min(e.frame_count() for e in experiments)
print("Frames per video:", batch.max_len)
os.makedirs(OUTPUT_DIR, exist_ok=True)

### Check the laser stripe

The detected stripe is blacked out in the image below. It should sit on the irradiated line. If it does not, use the optional ROI editor.

In [ ]:
for experiment in batch.experiments()[:2]:
    experiment.plot_frame_with_roi_masked()

In [ ]:
# Optional: move the stripe interactively (▲ / ▼ buttons, saved automatically)
EDIT_ROI = False

if EDIT_ROI:
    %matplotlib widget
    from DuckSeg.roi_editor import show_roi_editors
    show_roi_editors(batch.experiments())

## 4. Preprocessing

The transforms are applied to every frame before CellSAM. Tune them until most nuclei are outlined; the preview only processes a few frames, so it is quick. Details: [Tune the preprocessing](https://duckseg.readthedocs.io/en/latest/guides/preprocessing.html).

In [ ]:
%matplotlib inline
from DuckSeg.image_transforms import convert_to_u8, set_brightness, threshold_image, adjust_contrast
from DuckSeg.image_transformer import generate_masks_using_filters_and_transforms

transforms = [
    convert_to_u8,        # required: CellSAM needs 8-bit images
    set_brightness(20),   # scale every frame to the same mean brightness
    threshold_image(20),  # binary image: strong, clean signal for the nuclei
]

generate_masks_using_filters_and_transforms(
    batch,
    experiment_filter=[0],    # preview the first experiment
    frame_filter=2,           # first and last frame
    transforms=transforms,
    show_after_transform=[],  # e.g. [0, 1] to also show intermediate steps
)

## 5. Segment and track

CellSAM segments every frame, and each cell touching the stripe is followed through the video. This is the slow step (a few minutes on a GPU); results are cached next to the videos, so re-running is instant. After changing `transforms`, run once with `use_cache=False`.

In [ ]:
from DuckSeg.experiment_evaluator import evaluate_batch

vids = evaluate_batch(batch, transforms=transforms, use_cache=True)
print(f"{len(vids)} tracked cells")

## 6. Review the cells

Automatic filters remove cells with implausible mask areas. Then look at the cells: tick a checkbox to exclude a cell (a red frame and a red *excluded* label mark cells that are already excluded). Details: [Review and filter cells](https://duckseg.readthedocs.io/en/latest/guides/outlier_filtering.html).

In [ ]:
from DuckSeg.outlier_filters import outlier_vids, area_between, max_area_change_between_frames

range_modifier = 4                 # width of the scored band: 4 x the drawn stripe

filters = [
    area_between(10, 5000),                  # mask area in pixels, in every frame
    max_area_change_between_frames(0.3),     # no jumps > 30 % between frames
]
extra_manual_outliers = set()      # cell ids you always want to exclude, e.g. {"MSH2_001_3"}

outliers = outlier_vids(vids, extra_manual_outliers, filters)
print(f"{len(outliers)} of {len(vids)} cells removed by the filters")

In [ ]:
from DuckSeg.cell_video_visualizer import normalize_videos_for_display, show_video_grid_jupyter

manual_outliers = set()            # filled by the checkboxes below
display_vids = normalize_videos_for_display(vids, range_modifier=range_modifier)

for i in range(0, len(display_vids), 30):          # 30 cells per grid
    update = show_video_grid_jupyter(display_vids[i:i + 30], manual_outliers=manual_outliers)
    update(outliers, extra_manual_outliers)

In [ ]:
cell_filter = lambda vid: vid.id not in outliers and vid.id not in manual_outliers
kept = [vid for vid in vids if cell_filter(vid)]
print(f"Analysing {len(kept)} of {len(vids)} cells")

## 7. Recruitment curve

Mean over the kept cells with the standard error of the mean as a band. `Ratio` = inside / outside the band, `Diff` = inside − outside. With `baseline_correct=True` each cell is corrected to its first frame: `Diff(t) − Diff(0)` starts at 0, `Ratio(t) / Ratio(0)` starts at 1. See [Concepts](https://duckseg.readthedocs.io/en/latest/concepts.html).

In [ ]:
from DuckSeg.experiment_evaluator import ScoreType, plot_brightness_score

for score_type in [ScoreType.Ratio, ScoreType.Diff]:
    plot_brightness_score(batch, score_type=score_type, range_modifier=range_modifier,
                          cell_filter=cell_filter, baseline_correct=True,
                          save_path=os.path.join(OUTPUT_DIR, f"{CONDITION_NAME}_recruitment_{score_type.name.lower()}.png"))

In [ ]:
# Optional: every cell's trace in 3D, sorted by its mean (unusual traces stand out)
from DuckSeg.experiment_evaluator import calculate_brightness_score_for_videos, plot_all_vid_scores

ids, scores = calculate_brightness_score_for_videos(vids, score_type=ScoreType.Ratio, baseline_correct=True,
                                                    range_modifier=range_modifier, cell_filter=cell_filter)
plot_all_vid_scores(scores, ids)

## 8. Export

One row per frame and four columns per cell: raw `_normal` and `_ROI` intensities, and baseline-corrected `_diff` and `_ratio` scores (as in the plots). See [Export results](https://duckseg.readthedocs.io/en/latest/guides/export_results.html).

In [ ]:
from DuckSeg.video_exporter import export_cell_videos

csv_path = os.path.join(OUTPUT_DIR, f"{CONDITION_NAME}_brightness_scores.csv")
export_cell_videos(kept, csv_path, range_modifier=range_modifier, baseline_correct=True)   # same settings as the plots
print("Saved", csv_path)

## Next steps

* Analyse another condition: change `DATA_DIR` in section 1 and run again (each folder keeps its own cache).
* Compare conditions (one folder per condition) with `plot_brightness_scores`: [Compare conditions](https://duckseg.readthedocs.io/en/latest/guides/compare_conditions.html).
* Problems? See [Troubleshooting](https://duckseg.readthedocs.io/en/latest/troubleshooting.html).